<a href="https://colab.research.google.com/github/Fatema-S/BaseWebApp/blob/master/Applied_Deep_Learning_Age_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Setup and imports

In [ ]:
import sys
import shutil
import hashlib
import math

import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

import kagglehub

from sklearn.metrics import accuracy_score, f1_score
from sklearn.dummy import DummyClassifier

from tensorflow.keras import layers, models
import random

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Python version:", sys.version)
print("TensorFlow version:", tf.__version__)

# 2. Download and organise the dataset

In [ ]:
# Download the Kaggle dataset
path = kagglehub.dataset_download("frabbisw/facial-age")

KAGGLE_DIR = Path(path)

print("Kaggle dataset exists:", KAGGLE_DIR.exists())
print("Downloaded to:", KAGGLE_DIR)

# Create a separate original copy in Colab
ORIGINAL_DIR = Path("/content/datase_original")

shutil.copytree(
    KAGGLE_DIR,
    ORIGINAL_DIR,
    dirs_exist_ok=True
)

print("Original dataset copy created:", ORIGINAL_DIR.exists())

# Point to the folder containing the numbered age folders
DATA_DIR = ORIGINAL_DIR / "face_age" / "face_age"

print("Actual image folder:", DATA_DIR)
print("Image folder exists:", DATA_DIR.exists())

In [ ]:
age_folders = sorted(
    [
        folder for folder in DATA_DIR.iterdir()
        if folder.is_dir() and folder.name.isdigit()
    ],
    key=lambda folder: int(folder.name)
)

print("Number of age folders:", len(age_folders))
print("All age folders:", [f.name for f in age_folders[:]])

# 3. Initial dataset inspection

In [ ]:
# Get all files inside the age folders
all_files = [
    file
    for folder in age_folders
    for file in folder.iterdir()
    if file.is_file()
]

# Count file extensions
file_extensions = pd.Series(
    [file.suffix.lower() for file in all_files]
).value_counts()

print(file_extensions)

In [ ]:
# Get all folders with numeric names
age_folders = sorted(
    [item for item in DATA_DIR.iterdir()
     if item.is_dir() and item.name.isdigit()],
    key=lambda item: int(item.name)
)

# Count the number of files in each age folder
age_counts = {
    int(folder.name): len(list(folder.iterdir()))
    for folder in age_folders
}

# Create a pandas Series
age_distribution = pd.Series(age_counts)

# General statistics
print("Number of age folders:", len(age_distribution))
print("Total number of files:", age_distribution.sum())
print("Average files per age:", round(age_distribution.mean(), 2))

# Minimum
min_files = age_distribution.min()
folders_with_min = age_distribution[age_distribution == min_files]

print(f"\nMinimum files in an age folder: {min_files}")
print("Folder(s) with minimum number of files:")

for age, count in folders_with_min.items():
    print(f"  Age {age}: {count} files")

# Maximum
max_files = age_distribution.max()
folders_with_max = age_distribution[age_distribution == max_files]

print(f"\nMaximum files in an age folder: {max_files}")
print("Folder(s) with maximum number of files:")

for age, count in folders_with_max.items():
    print(f"  Age {age}: {count} files")

In [ ]:
# Store the dimensions and colour modes of all images
image_properties = []

for file in all_files:
    try:
        with Image.open(file) as img:
            image_properties.append({
                "filename": file.name,
                "width": img.width,
                "height": img.height,
                "mode": img.mode
            })
    except Exception as e:
        print("Could not open:", file, e)

# Convert results into a DataFrame
image_properties_df = pd.DataFrame(image_properties)

# Summarise the results
print("Image dimensions:")
print(image_properties_df.groupby(["width", "height"]).size())

print("\nColour modes:")
print(image_properties_df["mode"].value_counts())

# 4. Visual inspection and image quality

In [ ]:
# Store image sharpness scores
sharpness_scores = []

for file in all_files:
    img = cv2.imread(str(file))

    if img is None:
        print("Could not read:", file)
        continue

    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Calculate sharpness
    sharpness = cv2.Laplacian(gray, cv2.CV_64F).var()

    sharpness_scores.append({
        "filepath": str(file),
        "filename": file.name,
        "age": int(file.parent.name),
        "sharpness": sharpness
    })

# Create DataFrame
sharpness_df = pd.DataFrame(sharpness_scores)

print(sharpness_df.head())
print(sharpness_df["sharpness"].describe())

In [ ]:
least_sharp = sharpness_df.nsmallest(5, "sharpness")
most_sharp = sharpness_df.nlargest(5, "sharpness")

comparison = pd.concat([least_sharp, most_sharp])

print("Least sharp images:")
print(least_sharp[["filename", "age", "sharpness"]])

print("\nSharpest images:")
print(most_sharp[["filename", "age", "sharpness"]])

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 9))

for ax, (_, row) in zip(axes.flat, comparison.iterrows()):

    with Image.open(row["filepath"]) as img:
        ax.imshow(img)

    ax.set_title(
        f"{row['filename']}\n"
        f"Age: {row['age']}\n"
        f"Sharpness: {row['sharpness']:.1f}",
        fontsize=9
    )

    ax.axis("off")

fig.text(0.5, 0.95, "Least sharp",
         fontsize=13, fontweight="bold", ha="center")

fig.text(0.5, 0.48, "Sharpest",
         fontsize=13, fontweight="bold", ha="center")

plt.subplots_adjust(hspace=0.6, wspace=0.2)
plt.show()

In [ ]:
# Display 9 randomly selected images
fig, axes = plt.subplots(3, 3, figsize=(10, 10))

sample_files = np.random.choice(all_files, 9, replace=False)

for ax, file in zip(axes.flat, sample_files):
    with Image.open(file) as img:
        ax.imshow(img)
        ax.set_title(f"Age: {file.parent.name}")
        ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Choose the age folder you want to inspect
age_to_inspect = 80

# Get the selected folder
selected_folder = DATA_DIR / f"{age_to_inspect:03d}"

# Get all PNG images inside the folder
selected_images = sorted(selected_folder.glob("*.png"))

print(f"Age folder: {age_to_inspect}")
print(f"Number of images: {len(selected_images)}")

# Create a grid to display images
ncols = 7
nrows = (len(selected_images) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3))

axes = np.array(axes).flatten()

# Display images
for ax, file in zip(axes, selected_images):
    with Image.open(file) as img:
        ax.imshow(img)
        ax.set_title(file.name, fontsize=9)
        ax.axis("off")

# Hide unused spaces
for ax in axes[len(selected_images):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

# 5. Create the image DataFrame

In [ ]:
image_data = []

for file in all_files:
    image_data.append({
        "filepath": str(file),
        "age": int(file.parent.name)
    })

df = pd.DataFrame(image_data)

print(df.head())
print("Total images:", len(df))
print(df["age"].describe())

# 6. Duplicate detection and investigation

In [ ]:
def get_image_hash(filepath):
    with open(filepath, "rb") as file:
        return hashlib.md5(file.read()).hexdigest()


df["image_hash"] = df["filepath"].apply(get_image_hash)

print("Total images:", len(df))
print("Unique images:", df["image_hash"].nunique())
print("Duplicate files:", df["image_hash"].duplicated().sum())  # counts the number of extra occurrences of an image, not every image that belongs to a duplicate group.

In [ ]:
duplicate_hashes = df["image_hash"].value_counts()
duplicate_hashes = duplicate_hashes[
    duplicate_hashes > 1
].index

duplicates_df = df[
    df["image_hash"].isin(duplicate_hashes)
].copy()

duplicate_groups = list(
    duplicates_df.groupby("image_hash")
)

print("Duplicate groups:", len(duplicate_groups))
print("Files in duplicate groups:", len(duplicates_df))

Show the duplicates

In [ ]:

groups_per_page = 10

for start in range(0, len(duplicate_groups), groups_per_page):

    page_groups = duplicate_groups[start:start + groups_per_page]

    max_images = max(len(group) for _, group in page_groups)

    fig, axes = plt.subplots(
        len(page_groups),
        max_images,
        figsize=(3 * max_images, 3 * len(page_groups)),
        squeeze=False
    )

    for row, (image_hash, group) in enumerate(page_groups):

        for col, (_, image_row) in enumerate(group.iterrows()):

            filepath = image_row["filepath"]

            with Image.open(filepath) as img:
                axes[row, col].imshow(img.copy())

            axes[row, col].set_title(
                f"Age: {image_row['age']}\n"
                f"{Path(filepath).parent.name}/{Path(filepath).name}",
                fontsize=9
            )
            axes[row, col].axis("off")

        # Hide unused spaces
        for col in range(len(group), max_images):
            axes[row, col].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
for image_hash, group in duplicate_groups:
    ages = group["age"].unique()

    if len(ages) > 1:
        print("\nDifferent age labels:", ages)

        for _, row in group.iterrows():
            print(row["filepath"], "Age:", row["age"])

In [ ]:
# Recalculate duplicate statistics from the current groups

total_duplicate_groups = len(duplicate_groups)

conflicting_groups = [
    (image_hash, group)
    for image_hash, group in duplicate_groups
    if group["age"].nunique() > 1
]

same_age_groups = [
    (image_hash, group)
    for image_hash, group in duplicate_groups
    if group["age"].nunique() == 1
]

conflicting_files = sum(
    len(group) for _, group in conflicting_groups
)

same_age_extra_files = sum(
    len(group) - 1 for _, group in same_age_groups
)

print("Total duplicate groups:", total_duplicate_groups)
print("Conflicting-label groups:", len(conflicting_groups))
print("Same-age groups:", len(same_age_groups))

print("\nFiles in conflicting groups:", conflicting_files)
print("Extra copies in same-age groups:", same_age_extra_files)

print("\nExpected total exclusions:",
      conflicting_files + same_age_extra_files)

print("Extra duplicate occurrences:",
      df["image_hash"].duplicated().sum())

print("Expected exclusions using formula:",
      df["image_hash"].duplicated().sum() + len(conflicting_groups))

# 7. Remove unsuitable and duplicate images

In [ ]:
EXCLUDED_DIR = Path("/content/excluded_images")

SAME_AGE_DIR = EXCLUDED_DIR / "same_age_duplicates"
CONFLICTING_DIR = EXCLUDED_DIR / "conflicting_age_labels"
NON_FACE_DIR = EXCLUDED_DIR / "non_face_images"

for folder in [SAME_AGE_DIR, CONFLICTING_DIR, NON_FACE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

In [ ]:
exclusion_log = []

for image_hash, group in duplicate_groups:

    # Case 1: Same image has different age labels
    if group["age"].nunique() > 1:

        for _, row in group.iterrows():

            original_path = Path(row["filepath"])

            destination_folder = (
                CONFLICTING_DIR / original_path.parent.name
            )
            destination_folder.mkdir(parents=True, exist_ok=True)

            destination = destination_folder / original_path.name
            shutil.copy2(original_path, destination)

            exclusion_log.append({
                "filename": original_path.name,
                "age": row["age"],
                "category": "conflicting_age_labels",
                "original_path": str(original_path),
                "excluded_path": str(destination),
                "image_hash": image_hash
            })

    # Case 2: Same image has the same age label
    else:

        # Keep the first copy and exclude the additional copies
        for _, row in group.iloc[1:].iterrows():

            original_path = Path(row["filepath"])

            destination_folder = (
                SAME_AGE_DIR / original_path.parent.name
            )
            destination_folder.mkdir(parents=True, exist_ok=True)

            destination = destination_folder / original_path.name
            shutil.copy2(original_path, destination)

            exclusion_log.append({
                "filename": original_path.name,
                "age": row["age"],
                "category": "same_age_duplicates",
                "original_path": str(original_path),
                "excluded_path": str(destination),
                "image_hash": image_hash
            })

exclusion_df = pd.DataFrame(exclusion_log)

print(exclusion_df["category"].value_counts())
print("\nTotal images copied:", len(exclusion_df))

In [ ]:
exclusion_df.to_csv(
    EXCLUDED_DIR / "exclusion_log.csv",
    index=False
)

print("Exclusion log saved.")

In [ ]:
conflicting_hashes = [
    image_hash
    for image_hash, group in duplicate_groups
    if group["age"].nunique() > 1
]

In [ ]:
# Remove all images with conflicting age labels
clean_df = df[
    ~df["image_hash"].isin(conflicting_hashes)
].copy()

# Remove extra copies of images with the same age label
clean_df = clean_df.drop_duplicates(
    subset="image_hash",
    keep="first"
).copy()

clean_df = clean_df.reset_index(drop=True)

print("Original images:", len(df))
print("Cleaned images:", len(clean_df))
print("Images excluded:", len(df) - len(clean_df))
print("Remaining duplicate images:",
      clean_df["image_hash"].duplicated().sum())

# 8. Age groups and data distribution

In [ ]:
age_bins = [0, 12, 19, 39, 59, 110]

age_labels = [
    "Child",
    "Teenager",
    "Young adult",
    "Middle-aged adult",
    "Older adult"
]

clean_df["age_group"] = pd.cut(
    clean_df["age"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True
)

print(clean_df["age_group"].value_counts().sort_index())

In [ ]:
plt.figure(figsize=(12, 5))

plt.hist(clean_df["age"], bins=30, edgecolor="black")

plt.xlabel("Age")
plt.ylabel("Number of images")
plt.title("Age distribution after cleaning")

plt.show()

# 9. Training, validation and test splits

In [ ]:
# First, set aside 10% for testing
train_val_df, test_df = train_test_split(
    clean_df,
    test_size=0.10,
    random_state=RANDOM_SEED,
    stratify=clean_df["age_group"]
)

# Split the remaining 90% into training and validation
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.1667,
    random_state=RANDOM_SEED,
    stratify=train_val_df["age_group"]
)

In [ ]:
# create a folder to save the processed data
PROCESSED_DIR = Path("/content/processed_data")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_csv(PROCESSED_DIR / "train.csv", index=False)
val_df.to_csv(PROCESSED_DIR / "validation.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test.csv", index=False)

clean_df.to_csv(PROCESSED_DIR / "cleaned_dataset.csv", index=False)


In [ ]:
# Check the sizes
print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

print("Total:", len(train_df) + len(val_df) + len(test_df))
print("Cleaned dataset:", len(clean_df))

In [ ]:
# Check the age-group proportions
for name, split in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name} proportions:")
    print(split["age_group"].value_counts(normalize=True).sort_index())

# 10. Final checks and summary

In [ ]:
# Check that there are no duplicate images across splits
train_hashes = set(train_df["image_hash"])
val_hashes = set(val_df["image_hash"])
test_hashes = set(test_df["image_hash"])

print("Train-validation overlap:", len(train_hashes & val_hashes))
print("Train-test overlap:", len(train_hashes & test_hashes))
print("Validation-test overlap:", len(val_hashes & test_hashes))

# Check the saved files
print("\nSaved processed files:")
for file in sorted(PROCESSED_DIR.iterdir()):
    print(file.name, "-", file.stat().st_size, "bytes")

# Part 1: establishing baseline models

## Regression baseline — predict the mean age

In [ ]:
# Calculate the average age in the training set
mean_age = train_df["age"].mean()

# Predict this same age for every validation image
regression_predictions = np.full(
    len(val_df),
    mean_age
)

# Calculate MAE
regression_mae = mean_absolute_error(
    val_df["age"],
    regression_predictions
)

print("Mean age prediction:", round(mean_age, 2))
print("Regression baseline MAE:", round(regression_mae, 2), "years")

## classification baselines

In [ ]:

# Separate the training and validation labels
y_train = train_df["age_group"]
y_val = val_df["age_group"]

# 1. Majority-class baseline
majority_model = DummyClassifier(strategy="most_frequent")
majority_model.fit(np.zeros((len(y_train), 1)), y_train)

majority_predictions = majority_model.predict(
    np.zeros((len(y_val), 1))
)

majority_accuracy = accuracy_score(y_val, majority_predictions)
majority_f1 = f1_score(
    y_val, majority_predictions, average="macro"
)

# 2. Stratified-random baseline
random_model = DummyClassifier(strategy="stratified", random_state=42)
random_model.fit(np.zeros((len(y_train), 1)), y_train)

random_predictions = random_model.predict(
    np.zeros((len(y_val), 1))
)

random_accuracy = accuracy_score(y_val, random_predictions)
random_f1 = f1_score(
    y_val, random_predictions, average="macro"
)

# Display results
print("Majority-class baseline")
print("Accuracy:", round(majority_accuracy, 4))
print("Macro-F1:", round(majority_f1, 4))

print("\nStratified-random baseline")
print("Accuracy:", round(random_accuracy, 4))
print("Macro-F1:", round(random_f1, 4))

# Part 2: building your CNN models from scratch

## Set up TensorFlow and check the data

In [ ]:

# Import libraries
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image

# Check TensorFlow version
print("TensorFlow version:", tf.__version__)

# Check dataset sizes
print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

# Check the first few rows
display(train_df.head())


In [ ]:
# Check that the image paths still work


# Get the first training image path
first_image_path = train_df.iloc[0]["filepath"]

# Open the image
image = Image.open(first_image_path).convert("RGB")

# Display its properties
print("Image size:", image.size)
print("Image mode:", image.mode)
print("Actual age:", train_df.iloc[0]["age"])
print("Age group:", train_df.iloc[0]["age_group"])

# Display the image
plt.imshow(image)
plt.axis("off")
plt.show()


## Build the image-loading pipeline

In [ ]:

# Function to load and preprocess images
def load_images(dataframe):
    images = []

    for filepath in dataframe["filepath"]:
        # Open image and ensure RGB format
        image = Image.open(filepath).convert("RGB")

        # Convert image to numerical array
        image_array = np.array(image, dtype=np.float32)

        # Normalise pixel values between 0 and 1
        image_array = image_array / 255.0

        images.append(image_array)

    # Convert list of images into one NumPy array
    return np.array(images, dtype=np.float32)


# Load training and validation images
X_train = load_images(train_df)
X_val = load_images(val_df)

# Prepare regression labels
y_train_reg = train_df["age"].to_numpy(dtype=np.float32)
y_val_reg = val_df["age"].to_numpy(dtype=np.float32)

# Prepare classification labels
y_train_cls = train_df["age_group"].cat.codes.to_numpy()
y_val_cls = val_df["age_group"].cat.codes.to_numpy()

# Display shapes
print("Training images shape:", X_train.shape)
print("Validation images shape:", X_val.shape)

print("Training regression labels:", y_train_reg.shape)
print("Training classification labels:", y_train_cls.shape)

print("Minimum pixel value:", X_train.min())
print("Maximum pixel value:", X_train.max())


## Check classification label mapping

In [ ]:
# Display the mapping between age groups and numeric labels
age_group_mapping = {
    code: group
    for code, group in enumerate(train_df["age_group"].cat.categories)
}

print("Age group label mapping:")
for code, group in age_group_mapping.items():
    print(f"{code} = {group}")

# Count training examples in each class
print("\nTraining examples per age group:")
print(train_df["age_group"].value_counts().sort_index())

## Visualising some preprocessed images

In [ ]:
# Display 9 randomly selected training images
plt.figure(figsize=(10, 10))

for i in range(9):
    plt.subplot(3, 3, i + 1)

    plt.imshow(X_train[i])
    plt.title(
        f"Age: {int(y_train_reg[i])}\n"
        f"Group: {age_group_mapping[y_train_cls[i]]}"
    )
    plt.axis("off")

plt.tight_layout()
plt.show()

## Creating the regression CNN

In [ ]:

def build_regression_cnn():
    model = models.Sequential([
        layers.Input(shape=(200, 200, 3)),

        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D((2, 2)),

        # Reduce the feature maps without creating a huge vector
        layers.GlobalAveragePooling2D(),

        layers.Dense(128, activation="relu"),

        # Predict one numerical age
        layers.Dense(1)
    ])

    model.compile(
        optimizer="adam",
        loss="mean_absolute_error",
        metrics=["mean_absolute_error"]
    )

    return model


regression_model = build_regression_cnn()
regression_model.summary()


## Create a reproducibility function

In [ ]:

import random
import tensorflow as tf
import numpy as np

def set_random_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

# Test the function
set_random_seed(42)

print("Random seed set successfully!")
